In [ ]:
from pathlib import Path

import pandas as pd

from mp_stable_phases import fetch_stable_phases_for_elements, save_stable_phases_table

from build_datasheet import (
    build_datasheet,
    elemental_props_vs_host,
    optimized_dopant_list,
    save_datasheet,
)
from mp_structure_features import (
    fetch_and_featurize_phases,
    featurize_phases_with_prefix,
    save_structure_features,
)
from endmember_phases import (
    build_endmember_table,
    default_endmember_matminer_path,
    default_endmember_table_path,
    fetch_endmember_mp_metadata,
    save_endmember_table,
)


In [ ]:
ML_DIR = Path(".").resolve()
DATA_DIR = ML_DIR / "data"
REPO_ROOT = ML_DIR
DATA_DIR.mkdir(exist_ok=True)


In [ ]:
import os

# Set MP_API_KEY in your environment: https://materialsproject.org/api
# MP_API_KEY = os.environ.get("MP_API_KEY")
MP_API_KEY = ""
if not MP_API_KEY:
    raise EnvironmentError(
        "MP_API_KEY is not set. Export it before running this notebook, e.g.\n"
        '  export MP_API_KEY="your_key"'
    )


In [ ]:
from dataclasses import dataclass

@dataclass
class ElementData:
    Z: int               # atomic number
    r_atomic: float      # pm
    crystal: str         # polytype
    valency: int         # unit 
    chi: float           # Pauling scale


# Complete self-contained data for all elements (atomic number, metallic radius from XRD, crystal structure, valency, Pauling electronegativity χ)
ELEMENTS_DATA = {
    "H":  ElementData(1,   31, "molecular",  1,  2.20),
    "Li": ElementData(3,  152, "bcc",        1,  0.98),
    "Be": ElementData(4,  112, "hcp",        2,  1.57),
    "B":  ElementData(5,   85, "beta_rhombohedral", 3,  2.04), 
    "Na": ElementData(11,  186, "bcc",        1,  0.93),
    "Mg": ElementData(12,  160, "hcp",        2,  1.31),
    "Al": ElementData(13,  143, "fcc",        3,  1.61),
    "Si": ElementData(14,  111, "diamond",    4,  1.90),
    "K":  ElementData(19,  227, "bcc",        1,  0.82),
    "Ca": ElementData(20,  197, "fcc",        2,  1.00),
    "Sc": ElementData(21,  162, "hcp",        3,  1.36),
    "Ti": ElementData(22,  147, "hcp",        4,  1.54),
    "V":  ElementData(23,  134, "bcc",        5,  1.63),
    "Cr": ElementData(24,  128, "bcc",        3,  1.66),  # Cr  +3, +6
    "Mn": ElementData(25,  127, "complex",    2,  1.55),  # Mn  +2
    "Fe": ElementData(26,  126, "bcc",        2,  1.83),  # Fe  +2, +3
    "Co": ElementData(27,  125, "hcp",        2,  1.88),  # Co  +2
    "Ni": ElementData(28,  124, "fcc",        2,  1.91),  # Ni  +2
    "Cu": ElementData(29,  128, "fcc",        1,  1.90),  # Cu +1
    "Zn": ElementData(30,  134, "hcp",        2,  1.65),  # Zn +2
    "Ga": ElementData(31,  135, "ortho",      3,  1.81),
    "Ge": ElementData(32,  122, "diamond",    4,  2.01),
    "As": ElementData(33,  119, "rhombo",     5,  2.18),
    "Rb": ElementData(37,  248, "bcc",        1,  0.82),
    "Sr": ElementData(38,  215, "fcc",        2,  0.95),
    "Y":  ElementData(39,  180, "hcp",        3,  1.22),
    "Zr": ElementData(40,  160, "hcp",        4,  1.33),
    "Nb": ElementData(41,  146, "bcc",        5,  1.60),
    "Mo": ElementData(42,  139, "bcc",        6,  2.16),  # Mo  +6
    "Tc": ElementData(43,  136, "hcp",        7,  1.90),
    "Ru": ElementData(44,  134, "hcp",        3,  2.20),  # Ru  +3
    "Rh": ElementData(45,  134, "fcc",        3,  2.28),  # Rh  +3
    "Pd": ElementData(46,  137, "fcc",        2,  2.20),  # Pd  +2
    "Ag": ElementData(47,  144, "fcc",        1,  1.93),  # Ag +1
    "Cd": ElementData(48,  151, "hcp",        2,  1.69),  # Cd +2
    "In": ElementData(49,  167, "tetra",      3,  1.78),
    "Sn": ElementData(50,  140, "tetra",      4,  1.96),
    "Sb": ElementData(51,  140, "rhombo",     5,  2.05),
    "Cs": ElementData(55,  265, "bcc",        1,  0.79),
    "Ba": ElementData(56,  222, "bcc",        2,  0.89),
    "Hf": ElementData(72,  159, "hcp",        4,  1.30),
    "Ta": ElementData(73,  146, "bcc",        5,  1.50),
    "W":  ElementData(74,  139, "bcc",        6,  2.36),  # W  +6
    "Re": ElementData(75,  137, "hcp",        7,  1.90),
    "Os": ElementData(76,  135, "hcp",        4,  2.20),  # Os  +4
    "Ir": ElementData(77,  136, "fcc",        3,  2.20),  # Ir  +3
    "Pt": ElementData(78,  139, "fcc",        2,  2.28),  # Pt  +2
    "Au": ElementData(79,  144, "fcc",        1,  2.54),  # Au +1
    "Hg": ElementData(80,  151, "rhombo",     2,  2.00),  # Hg +2
    "Bi": ElementData(83,  155, "rhombo",     5,  2.02),
    "La": ElementData(57,  187, "dhcp",       3,  1.10),
    "Ce": ElementData(58,  182, "fcc",        3,  1.12),
    "Pr": ElementData(59,  182, "hcp",        3,  1.13),
    "Nd": ElementData(60,  181, "hcp",        3,  1.14),
    "Sm": ElementData(62,  180, "rhombo",     3,  1.17),
    "Eu": ElementData(63,  208, "bcc",        2,  0.63),  # Eu  +2
    "Gd": ElementData(64,  180, "hcp",        3,  1.20),
    "Tb": ElementData(65,  177, "hcp",        3,  1.22),
    "Dy": ElementData(66,  178, "hcp",        3,  1.22),
    "Ho": ElementData(67,  176, "hcp",        3,  1.23),
    "Er": ElementData(68,  176, "hcp",        3,  1.24),
    "Tm": ElementData(69,  176, "hcp",        3,  1.25),
    "Yb": ElementData(70,  194, "fcc",        2,  1.10),  # Yb  +2
    "Lu": ElementData(71,  174, "hcp",        3,  1.27),

    "Se": ElementData(34, 116, "trigonal",     6,  2.55),
    "Tl": ElementData(81, 170, "hcp",          3,  1.62),
    "Pb": ElementData(82, 175, "fcc",          4,  2.33),
}

def get_element_data(sym: str) -> ElementData:
    sym = sym.strip().title()
    data = ELEMENTS_DATA.get(sym)
    if data is None:
        raise ValueError(f"Element '{sym}' not found in ELEMENTS_DATA")
    return data


def element_radii_lookup() -> dict[str, float]:
    """Radii (pm) from ELEMENTS_DATA for MP/matminer helpers."""
    return {sym: float(data.r_atomic) for sym, data in ELEMENTS_DATA.items()}


In [ ]:
host_element = 'Li'


In [ ]:
dopant_list = optimized_dopant_list(REPO_ROOT)
print(f"{len(dopant_list)} optimized dopants: {dopant_list}")

elemental_props_df = elemental_props_vs_host(
    dopant_list,
    host_element,
    ELEMENTS_DATA,
    get_element_data=get_element_data,
)
elemental_props_df.to_csv(DATA_DIR / "elemental_props_vs_li.csv", index=False)
print(f"Saved elemental properties vs Li: {DATA_DIR / 'elemental_props_vs_li.csv'}")
elemental_props_df


In [ ]:
stable_phases_df = fetch_stable_phases_for_elements(
    dopant_list,
    host_element=host_element,
    api_key=MP_API_KEY,
    e_hull_max=0.02,
    strict_less_than=True,
    selection_mode="min_nsites",
    require_experimental=True,
    prefer_is_stable=False,
    use_rt_mp_catalog=False,
    allow_theoretical_fallback=False,
    include_manual_radii=False,
)

save_stable_phases_table(
    stable_phases_df,
    DATA_DIR / "dopant_stable_experimental_phases_mp.csv",
)
stable_phases_df


In [ ]:
phase_structures, matminer_features_df = fetch_and_featurize_phases(
    stable_phases_df,
    api_key=MP_API_KEY,
)

save_structure_features(
    matminer_features_df,
    DATA_DIR / "structure_matminer_features.csv",
)
matminer_features_df.head()


In [ ]:
endmember_table = build_endmember_table()
endmember_phases_df = fetch_endmember_mp_metadata(endmember_table, api_key=MP_API_KEY)
endmember_phases_path = save_endmember_table(
    endmember_phases_df,
    default_endmember_table_path(ML_DIR),
)
print(f"Saved {len(endmember_phases_df)} endmember rows to {endmember_phases_path}")
endmember_phases_df


In [ ]:
endmember_featurize_df = endmember_table.dropna(subset=["endmember_material_id"]).copy()
endmember_featurize_df = endmember_featurize_df.rename(
    columns={"endmember_material_id": "material_id"}
)
endmember_featurize_df["role"] = "endmember"
_, endmember_matminer_df = featurize_phases_with_prefix(
    endmember_featurize_df[["element", "material_id", "role", "endmember_formula"]],
    api_key=MP_API_KEY,
    prefix="endmember_",
    extra_id_cols=("endmember_formula",),
)
endmember_matminer_path = save_structure_features(
    endmember_matminer_df,
    default_endmember_matminer_path(ML_DIR),
)
print(f"Saved endmember matminer features to {endmember_matminer_path}")
endmember_matminer_df.head()


In [ ]:
datasheet_df = build_datasheet(
    REPO_ROOT,
    mp_phases_path=DATA_DIR / "dopant_stable_experimental_phases_mp.csv",
    elemental_props_df=elemental_props_df,
)

datasheet_path = save_datasheet(datasheet_df, DATA_DIR)
print(f"Saved: {datasheet_path} ({len(datasheet_df)} dopants)")
datasheet_df.head()


In [ ]:
datasheet_df.head(30)
